# PINN v2_8 workflow diagram

Renders a publication-ready flowchart of the North Cotabato PINN v2_8 pipeline
(data preprocessing → model architecture → training → evaluation) using only matplotlib
(no graphviz). Saves `figures/v2_8_workflow.png` and `figures/v2_8_workflow.pdf`.

Edit the box text / positions in the drawing cell to adapt it (e.g. for the South Mindanao run).

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch, Rectangle

FIG_DIR = Path.cwd().parent / "figures"
FIG_DIR.mkdir(exist_ok=True)

C = {"prep": "#2e7d32", "arch": "#1565c0", "train": "#e65100", "eval": "#6a1b9a"}
TINT = {"prep": "#e3f2e4", "arch": "#e3eefb", "train": "#fdece0", "eval": "#f0e5f7",
        "phys": "#d7e9ff", "io": "#ffffff"}


def box(ax, cx, cy, w, h, text, fc, ec="#37474f", fs=8.5, bold=False):
    ax.add_patch(FancyBboxPatch((cx - w / 2, cy - h / 2), w, h,
                 boxstyle="round,pad=0.15,rounding_size=1.1",
                 linewidth=1.1, edgecolor=ec, facecolor=fc, zorder=3))
    ax.text(cx, cy, text, ha="center", va="center", fontsize=fs,
            fontweight="bold" if bold else "normal", zorder=4, color="#212121")
    return dict(cx=cx, cy=cy, top=cy + h / 2, bot=cy - h / 2, l=cx - w / 2, r=cx + w / 2)


def arrow(ax, p, q, color="#455a64", lw=1.5, rad=0.0):
    ax.add_patch(FancyArrowPatch(p, q, arrowstyle="-|>", mutation_scale=13, lw=lw,
                 color=color, connectionstyle=f"arc3,rad={rad}", zorder=2, shrinkA=1, shrinkB=1))


def vdown(ax, a, b, **kw):
    arrow(ax, (a["cx"], a["bot"]), (b["cx"], b["top"]), **kw)


def band(ax, y0, y1, color, label):
    ax.add_patch(Rectangle((1, y0), 98, y1 - y0, facecolor=color, alpha=0.07,
                 edgecolor=color, linewidth=1.0, zorder=0))
    ax.text(4.2, (y0 + y1) / 2, label, rotation=90, va="center", ha="center",
            fontsize=12.5, fontweight="bold", color=color, zorder=1)

In [ ]:
fig, ax = plt.subplots(figsize=(13.5, 19.5))
ax.set_xlim(0, 100); ax.set_ylim(6, 150); ax.axis("off")
ax.set_title("PINN v2_8 workflow — earthquake-induced landslide susceptibility (North Cotabato)",
             fontsize=14, fontweight="bold", pad=14)

CL = 55  # center line for single-column boxes

# ---- 1 · preprocessing ----
band(ax, 123, 149, C["prep"], "1 · DATA PREPROCESSING")
p0 = box(ax, CL, 145, 74, 6, "SU_17 training GeoPackage\nslope-unit polygons + binary landslide label", TINT["io"], bold=True)
p1 = box(ax, CL, 137.5, 84, 6.6, "preprocessing_v2( )\ndrop leaked cols  ·  slope filter \u2265 10\u00b0  ·  median impute + missingness-indicator flags", TINT["prep"])
p2 = box(ax, CL, 130.5, 84, 6.6, "GA-EN feature selection  \u2192  8 numeric + type\nsoil_texture_idx derived from Clay/Silt/Sand (USDA texture triangle)", TINT["prep"])
p3 = box(ax, CL, 124.5, 84, 5.4, "per-fold (inside CV, leakage-safe):  log1p(skew>1) + clip(1/99), physics excluded  \u2192  transform manifest", TINT["prep"])
vdown(ax, p0, p1); vdown(ax, p1, p2); vdown(ax, p2, p3)

# ---- 2 · architecture ----
band(ax, 58, 121, C["arch"], "2 · MODEL ARCHITECTURE  (LandslideRainFallV3, PINN)")
a_in = box(ax, CL, 116, 80, 6, "Encoded inputs:  8 numeric (NormalizationLayer)  +  type (embedding)  +  soil_texture_idx", TINT["io"], bold=True)
g1 = box(ax, 27, 106, 30, 8, "Geotechnical MLP\n8-deep Dense\nBatchNorm + LeakyReLU\n(excludes Prc_mean)", TINT["arch"], fs=8)
g2 = box(ax, 27, 97, 20, 4.4, "Dense(2)", TINT["arch"])
gc = box(ax, 18, 90, 15, 4.6, "Cohesion", TINT["phys"])
gi = box(ax, 37, 90, 17, 4.6, "Internal\nfriction", TINT["phys"], fs=8)
h1 = box(ax, 78, 106, 34, 8, "soil_texture_idx \u2192\nHydraulicConductivityLayerV3\nlearnable K per USDA class", TINT["arch"], fs=8)
h2 = box(ax, 77, 96, 38, 6.6, "WetnessLayer\n[Prc, ContribArea, SoilThc, Slope, K] \u2192 m  (wetness, [0,1])", TINT["phys"], fs=8)
d1 = box(ax, 47, 84, 54, 7.2, "DisplacementLayerRainFall\n[cohesion, friction, Slope, PGA, BUK, m]  \u2192  FOS \u00b7 displacement \u00b7 critical acceleration", TINT["phys"], fs=8)
d2 = box(ax, 47, 76, 40, 5.6, "NewmarkActivation(threshold=10)\n\u2192  physics_prob", TINT["phys"], bold=True, fs=8.5)
r1 = box(ax, 88, 84, 21, 8, "Residual DNN\nfeatures \u2192 tanh \u00d7 3\n(bounded \u00b13 logits)", TINT["arch"], fs=8)
comb = box(ax, 50, 67.5, 74, 5.6, "final_logit = logit(physics_prob) + bounded residual   \u2192   sigmoid", TINT["arch"], fs=8.5)
a_out = box(ax, CL, 60.8, 80, 5.4, "Outputs:  final_head (susceptibility)   +   physics_prob (auxiliary supervision head)", TINT["io"], bold=True)

arrow(ax, (a_in["cx"] - 12, a_in["bot"]), (g1["cx"], g1["top"]), rad=0.15)
arrow(ax, (a_in["cx"] + 12, a_in["bot"]), (h1["cx"], h1["top"]), rad=-0.15)
vdown(ax, g1, g2)
arrow(ax, (g2["cx"], g2["bot"]), (gc["cx"], gc["top"]), rad=0.1)
arrow(ax, (g2["cx"], g2["bot"]), (gi["cx"], gi["top"]), rad=-0.1)
vdown(ax, h1, h2)
arrow(ax, (gc["cx"], gc["bot"]), (d1["l"] + 6, d1["top"]))
arrow(ax, (gi["cx"], gi["bot"]), (d1["l"] + 14, d1["top"]))
arrow(ax, (h2["cx"], h2["bot"]), (d1["r"] - 10, d1["top"]), rad=0.05)
vdown(ax, d1, d2)
arrow(ax, (d2["cx"], d2["bot"]), (comb["cx"] - 10, comb["top"]))
arrow(ax, (r1["cx"], r1["bot"]), (comb["r"] - 6, comb["top"]), rad=-0.1)
arrow(ax, (a_in["r"] - 4, a_in["cy"]), (r1["cx"], r1["top"]), color="#90a4ae", rad=-0.3, lw=1.1)
vdown(ax, comb, a_out)

# ---- 3 · training ----
band(ax, 33, 56, C["train"], "3 · TRAINING")
t1 = box(ax, CL, 52, 84, 5.6, "StratifiedKFold(5, seed 42)  \u00b7  per-fold transforms + NormalizationLayer adapted on that fold only", TINT["train"])
t2 = box(ax, 30, 44.5, 40, 6.6, "Loss:  Dice-CrossEntropy (final_head)\n+ BinaryCE (physics_prob, weight 0.7)", TINT["train"], fs=8)
t3 = box(ax, 76, 44.5, 42, 6.6, "Adam 1e-4  \u00b7  class_weight {0:1, 1:5}\nEarlyStopping(val AUC, patience 5) \u00b7 200 ep \u00b7 bs 128", TINT["train"], fs=8)
t4 = box(ax, CL, 36.5, 84, 5.4, "Save  fold-{1..5}-model-v3.keras   +   out-of-fold predictions   +   per-fold manifests", TINT["train"], bold=True)
arrow(ax, (t1["cx"], t1["bot"]), (t2["cx"], t2["top"]), rad=0.12)
arrow(ax, (t1["cx"], t1["bot"]), (t3["cx"], t3["top"]), rad=-0.12)
arrow(ax, (t2["cx"], t2["bot"]), (t4["cx"] - 8, t4["top"]), rad=-0.12)
arrow(ax, (t3["cx"], t3["bot"]), (t4["cx"] + 8, t4["top"]), rad=0.12)

# ---- 4 · evaluation ----
band(ax, 8, 31, C["eval"], "4 · EVALUATION")
e0 = box(ax, CL, 27, 84, 5.4, "Out-of-fold predictions  (each row scored by the fold that did NOT train on it)", TINT["eval"], bold=True)
e1 = box(ax, 20.5, 17, 26, 11, "Discrimination\nROC / AUC\nconfusion matrix\nPrecision-Recall", TINT["eval"], fs=8)
e2 = box(ax, 42, 17, 26, 11, "Calibration\nBrier score\nreliability curve\nsuccess-rate curve", TINT["eval"], fs=8)
e3 = box(ax, 63.5, 17, 26, 11, "Physics & maps\nFOS/coh/ifi/disp.\nsusceptibility maps\nK vs literature", TINT["eval"], fs=8)
e4 = box(ax, 85, 17, 26, 11, "Robustness\nSHAP importance\nfold-ensemble unc.\nfalse-positive analysis", TINT["eval"], fs=8)
for e in (e1, e2, e3, e4):
    arrow(ax, (e0["cx"], e0["bot"]), (e["cx"], e["top"]), lw=1.2)

# ---- cross-stage spine ----
for a, b, col in [(p3, a_in, "#263238"), (a_out, t1, "#263238"), (t4, e0, C["train"])]:
    arrow(ax, (a["cx"], a["bot"]), (b["cx"], b["top"]), color=col, lw=2.0)

fig.tight_layout()
fig.savefig(FIG_DIR / "v2_8_workflow.png", dpi=200, bbox_inches="tight")
fig.savefig(FIG_DIR / "v2_8_workflow.pdf", bbox_inches="tight")
print("saved ->", FIG_DIR / "v2_8_workflow.png", "and .pdf")
plt.show()